In [ ]:
"""
Project 04: Hospital Readmission
Step 02: Model Training and Calibration
"""

In [ ]:
import warnings

In [ ]:
warnings.filterwarnings("ignore")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib

In [ ]:
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pickle
import os
import time
from tqdm import tqdm
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from lightgbm import LGBMClassifier
from sklearn.calibration import CalibratedClassifierCV
from imblearn.over_sampling import SMOTE

In [ ]:
BASE = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
PROCESSED = os.path.join(BASE, "data", "processed")
MODELS = os.path.join(BASE, "models")

In [ ]:
os.makedirs(PROCESSED, exist_ok=True)
os.makedirs(MODELS, exist_ok=True)

In [ ]:
print("=" * 60)
print("STEP 02: Model Training and Calibration")
print("=" * 60)
start_time = time.time()

In [ ]:
# ─── 1. Load and Prepare Data ───
print("\n[1/4] Loading and preparing data...")
df = pd.read_csv(os.path.join(PROCESSED, "readmission_cleaned.csv"))
print(f"  Loaded: {df.shape[0]} records, {df.shape[1]} columns")

In [ ]:
df.drop(
    columns=["encounter_id", "patient_nbr", "readmitted"], inplace=True, errors="ignore"
)

In [ ]:
le_dict = {}
cat_cols = df.select_dtypes(include=["object"]).columns.tolist()
print(f"  Encoding {len(cat_cols)} categorical columns...")
for col in tqdm(cat_cols, desc="  Label Encoding"):
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    le_dict[col] = le

In [ ]:
X = df.drop("readmitted_30d", axis=1)
y = df["readmitted_30d"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"  Train: {X_train.shape[0]} | Test: {X_test.shape[0]}")

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
with open(os.path.join(MODELS, "scaler.pkl"), "wb") as f:
    pickle.dump(scaler, f)
with open(os.path.join(MODELS, "label_encoders.pkl"), "wb") as f:
    pickle.dump(le_dict, f)
print("  Saved: scaler.pkl, label_encoders.pkl")

In [ ]:
# ─── 2. SMOTE Oversampling ───
print("[2/4] Applying SMOTE for class imbalance...")
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train_scaled, y_train)
print(f"  After SMOTE: {X_train_res.shape[0]} samples (from {X_train.shape[0]})")

In [ ]:
# ─── 3. Hyperparameter Optimization ───
print("[3/4] Training LightGBM with hyperparameter search...")
param_grid = {
    "n_estimators": [100, 200, 500],
    "learning_rate": [0.01, 0.05, 0.1],
    "num_leaves": [31, 64, 128],
    "max_depth": [5, 10, -1],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
}

In [ ]:
lgb = LGBMClassifier(random_state=42, verbose=-1)
rs = RandomizedSearchCV(
    lgb, param_grid, n_iter=10, cv=3, scoring="roc_auc", n_jobs=-1, random_state=42
)

In [ ]:
for i in tqdm(range(1), desc="  RandomizedSearchCV"):
    rs.fit(X_train_res, y_train_res)

In [ ]:
best_lgb = rs.best_estimator_
print(f"  Best Parameters: {rs.best_params_}")
print(f"  Best CV AUROC: {rs.best_score_:.4f}")

In [ ]:
# ─── 4. Calibration and Baseline ───
print("[4/4] Calibrating models and saving...")
calibrated_lgb = CalibratedClassifierCV(best_lgb, method="sigmoid", cv=None)
calibrated_lgb.fit(X_train_res, y_train_res)

In [ ]:
with open(os.path.join(MODELS, "lgbm_calibrated.pkl"), "wb") as f:
    pickle.dump(calibrated_lgb, f)
print("  Saved: lgbm_calibrated.pkl")

In [ ]:
# Baseline logistic regression
print("  Training Logistic Regression baseline...")
lr = LogisticRegression(max_iter=1000, random_state=42, class_weight="balanced")
lr.fit(X_train_scaled, y_train)

In [ ]:
with open(os.path.join(MODELS, "logistic_baseline.pkl"), "wb") as f:
    pickle.dump(lr, f)
print("  Saved: logistic_baseline.pkl")

In [ ]:
elapsed = time.time() - start_time
print(f"\n{'=' * 60}")
print(f"STEP 02 COMPLETE | Time: {elapsed:.1f}s")
print(f"{'=' * 60}")